# Fine-tuning of the U-Net on the MDT records of `munich_ablation`

**Purpose.** This notebook fine-tunes all parameters of the pretrained U-Net on the train
records.

**Inputs.** `lab/outputs/munich_ablation/models/pretrained*.pt` and the files of
`lab/data/munich_ablation/processed/`.

**Method.** The predicted RSRP of a received (record, cell) pair is
$P_{\mathrm{RS},c} + \hat{G}_c(g_r)$, where $g_r$ is the grid cell that holds the
reported position of record $r$. The loss is the mean absolute error (MAE) of the RSRP in dB
over the train pairs. A record outside the grid has no pair. The kept parameters have the
smallest MAE on the validation records.

**Outputs.** `lab/outputs/munich_ablation/models/fine_tuned*.pt` and `lab/outputs/munich_ablation/histories/`.

**Requirements.** Run `03_pretraining.ipynb` first.

## 1. Setup

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from experiments import (
    channels_without,
    fine_tune_model,
    history_figure,
    load_model,
    load_scenario,
    new_model,
    save_model,
)
from radio_calibration.evaluation import ResourceMonitor
from utils import load_config, save_figure, save_table, setup_notebook

cfg = load_config("scenario=munich_ablation")
device = setup_notebook(cfg)
data = load_scenario(cfg)

## 2. Fine-tuning

The scenario `munich_ablation` also fine-tunes each variant of `scenario.ablations`. A variant
with `pretraining: false` starts from new parameters.

In [ ]:
histories, resources = {}, {}
runs = {"fine_tuned": ("pretrained", None)}
for variant, spec in cfg.scenario.ablations.items():
    runs[f"fine_tuned_{variant}"] = (f"pretrained_{variant}" if spec.pretraining else None, spec)
for model_name, (source, spec) in runs.items():
    if source is not None:
        model = load_model(data, source, device)
    else:
        model = new_model(cfg, data, device, channels_without(spec.drop))
    with ResourceMonitor() as monitor:
        history = fine_tune_model(cfg, data, model)
    save_model(data, model_name, model, history)
    histories[model_name] = history
    resources[model_name] = monitor.summary()

In [ ]:
fig = history_figure(histories)
save_figure(fig, "fine_tuning_history")
plt.show()

**Fig. 1.** Fine-tuning. (a) MAE of the train pairs in each epoch. (b) MAE of the
validation pairs.

In [ ]:
save_table(
    pd.DataFrame(
        {
            model_name: {
                "Smallest validation MAE (dB)": history["validation_loss"].min(),
                "Epoch of the smallest validation MAE": history["validation_loss"].idxmin() + 1,
                "Mean time of one step (s)": history["step_seconds"].mean(),
            }
            for model_name, history in histories.items()
        }
    )
    .T.style.set_caption("Table 1. Result of the fine-tuning.")
    .format(precision=4),
    "fine_tuning_result",
)

In [ ]:
save_table(
    pd.DataFrame(resources)
    .T.style.set_caption("Table 2. Resource use of the fine-tuning.")
    .format(precision=1),
    "fine_tuning_resources",
)